Part 2b - notes

The model is $y_i = f(x_i) + \epsilon_i$

where $f(x_i)$ is the polynomial that we are trying to fit and $\epsilon_i$ is the random error. The expectation consists of a bias-term, a variance-term and a noise-term. The expectation $E[.] means averaging over the ramdomness in the errors. In other words, imagine repeatedly generating new datasets with the sam $x_i$'s but different random noise $\epsilon_i$. The expectation tells us what we would get on average over all those possible noise realizations. 

The bias-term: 
How far the average polynomial, fitted from many datasets, is from the true underlying function $f(x)$. A polynomial that is too simple may systematically miss the true curve, giving high bias. 

The variance-term: How much the fitted polynomial changes from one dataset to another because each dataset contains different random errors. A very flexible polynomial degree can wiggle dramatically depending on the perticular data, giving high variance. 

Noise-term: The randomness that is intrinsic to the observations and cannot be eliminated just by choosing a better polynomial. If $y = f(x) + \epsilon$ and $E[\epsilon^2] = \sigma^2$, this is the irreducible error. 

In short: Expectation averages over all the different possible noisy datasets. Bias measures systematic error, variance measures sensitivity to the particular dataset, and $\sigma^2$ measures the unavoidable noise in the observations. 



From lecture notes:
Bias - How far the average prediction is from the truth: The error built into the model's simplifying assumptions. A straight line through curved data has large bias, and no amount of data removes it. Decreases with model complexity. 

Variance - How much the prediction moves when the training set is redrawn. A flexible model chases the noise of whatever sample it got, so its prediction differ wildly between training sets. Increases with model complexity.

$\sigma^2$ - The irreducible error: no model can predict the noise term. A hard floor under the test error.

The test error is the sum of them all. It falls when the bias falls, then rises when the variance takes over. 


part 3.2)

The graph creates a U-shape, where the MSE is bigger for lower and higher polynomial degrees. The high MSE at low degrees is due to the model being underfitted. It is not complex enough to make a good fit. The bias dominates in this region. This means that all of the predicted values deviates from the true values, because the approximated function does not have enough parameters to resemble the true function. 

For high polynomial degrees, we see an increase in MSE as well. This is due to overfitting. The variance dominates here. The complex model starts fitting to the noise in the training data and the variance between the predicted values increases. 

In the middle region of degrees 4-10 has the lowest MSE- This means that the optimal polynomial degree is within this range. From the figure and measurements, we see that the MSE is lowest when the polynomial has 8 degrees.

Part 3.3)

For an increase in data points, the variance decreases noticably for high polynomial degrees. This seems reasonable because, since we increase the number of data points so that the distance between them decreases as they fill up more of the area where the data lies. The bias dominated area does not change much, but the variance dominated area is heavily affected by the number of data points. The bias generally stays the same. This also makes sense as the bias is not affected by the number of data datapoints. Bias is only affected by systematic errors, as the bias only decreases if we improve the fit of the model. 

part 3.4)

This shifts the complexity knob from polynomial degree to the regularization strength $\lambda$, with fixed degree at 12 (so the model is maximally flexible in terms of basis functions, and alone $\lambda$ controls effective complexity).

Increasing $\lambda$ shrinks the model's coefficients toward zero, which directly reduces its variance. At small $\lambda$ the degree-12 model is highly flexible and overfits the specific noise in each bootstrap resample, producing wildly different fitted curves across resamples. As $\lambda$ grows, that flexibility is progressively constrained, so fits across resamples converge and variance falls. often by orders of magnitude over the low-to-moderate $\lambda$-range.

Over this same range, the squared bias stays rouhly flat. The true function is smooth, so removing the model's ability to chase high-frequency noise costs little in terms of how well it can represent the actual signal. Because error $\approx$ bias$^2$ + variance, and variance is dropping while bias$^2$ is essentially unchanged, total test error falls in step with variance. This is the free regularization regime. 

This doesn't continue forever. At large $\lambda$, shrinkage becomes strong enough to also suppress the coefficients the model needs to represent the true (smooth) function, and bias$^2$ starts rising. Test error eventually turns around and increases too, once bias$^2$ dominates.


$\lambda$ attacks variance, and it does so at the cost of bias$^2$, but only once $\lambda$  is large enough.

For small-to-moderate $\lambda$, the trade is essentially free (variance falls, bias stays put). The cost only kicks in once $\lambda$ overshoots into a regime where the model is too constrained to fit the true signal, at which point rising bias$^2$ outweighs any further variance reduction. The optimal $\lambda$ sits at the boundary between these two regimes — where variance has already been squeezed out but bias$\^2$ hasn't yet started to climb.


part 4.2)

Expectations: The two methods usually agree closely or land within one degree of each other. CV and bootstrap are both estimating the same generalization error via resampling, just with different resampling schemes (partition-without-replacement vs. resample-with-replacement). Small discrepancies are normal and mostly reflect that both are noisy estimators, especially at small n.

The two methods point to the same degree window for polynomial degrees up to 10. For higher degrees than this, the bootstrap method is dominated by variance and therefore skyrockets. Choosing a specific number of k-folds affects the variance of the error estimate. Using a smaller k (5 or 10) increases the difference between training sets, which paradoxically stabilizes the variance of the resulting MSE estimate, though it introduces more bias. This is because the training sets for each fold share a very little percentage of the same data. If you increase the number of k-folds, the training sets share a bigger portion of the data and the variance will increase.  Smaller K uses folds with less overlap in training data, making the individual performance evaluations more independent and less prone to the high statistical covariance seen in high-K setups

part 4.3)

The scaling must be fit inside each fold because the StandardScaler learns its mean and standard deviation from the data it is fit on. If you fit the scaler once on the full dataset before splitting into folds, the scaling parameters of each fold's training set ar computed using information from that fold's test set too. A form of data leakage. This makes the CV estimate optimistically biased, because the fold is no longer being evaluated on genuinely unseen data. Fitting StandardScaler fresh inside each fold (which make_pipeline handles automatically when the whole pipeline is .fit() inside the loop, as done above) keeps each fold's test data completely untouched during training.

part 4.4)

Stability of selected λ: as k increases toward LOO (k=n), each fold's training set is nearly the full dataset (only one point held out), so training sets across folds become almost identical — this tends to reduce the variance of the CV estimate itself (less sensitive to how the data happens to be split), and the selected λ across k=5, k=10, k=n should generally converge to similar values, though k=5 may show a bit more fold-to-fold variability given its coarser partition (only 20 points/fold at n=100).
Computational cost: cost scales directly with k, since you refit the model k times per λ. LOO with n=100 means 100 fits per λ value — with 100 λ values in your grid, that's 10,000 model fits, versus 500 for k=5. LOO is the most expensive by far and this cost grows linearly with n, which is the main practical reason k=5 or k=10 are preferred defaults in practice — LOO's stability gain rarely justifies its cost except for very small datasets.